In [3]:
# Cell: Load and validate the final dataset
!pip install seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

# Load datasets
dc_final = pd.read_csv('/Users/zdiener/data-center-classification/data/processed/datacenters_with_census_tracts.csv')
communities = pd.read_csv('/Users/zdiener/data-center-classification/data/raw/2.0-communities.csv')

print("="*60)
print("FINAL DATASET VALIDATION")
print("="*60)

print(f"\nDataset Dimensions:")
print(f"  Data centers: {len(dc_final)} rows × {len(dc_final.columns)} columns")
print(f"  Communities: {len(communities)} rows × {len(communities.columns)} columns")

print(f"\nData Sources Breakdown:")
print(dc_final['source'].value_counts())

print(f"\nMatch Type Breakdown:")
if 'match_type' in dc_final.columns:
    print(dc_final['match_type'].value_counts())

FINAL DATASET VALIDATION

Dataset Dimensions:
  Data centers: 2536 rows × 53 columns
  Communities: 74134 rows × 136 columns

Data Sources Breakdown:
source
frac_supplemental    1294
osdc_primary         1242
Name: count, dtype: int64

Match Type Breakdown:
match_type
frac_only    1294
osdc_only     877
matched       365
Name: count, dtype: int64


/var/folders/wy/tbdpl_451258rc4j_fzpf5x80000gn/T/ipykernel_76080/678606521.py:14: DtypeWarning: Columns (0: Identified as disadvantaged due to tribal overlap, 1: Income data has been estimated based on geographic neighbor income, 2: Does the tract have at least 35 acres in it?, 3: Tract experienced historic underinvestment, 4: Is there at least one abandoned mine in this census tract?, 5: Names of Tribal areas within Census tract) have mixed types. Specify dtype option on import or set low_memory=False.
  communities = pd.read_csv('/Users/zdiener/data-center-classification/data/raw/2.0-communities.csv')


In [4]:
# Cell: Validate Census Tract Geocoding

print("="*60)
print("CENSUS TRACT GEOCODING VALIDATION")
print("="*60)

# Geocoding success rate
total = len(dc_final)
with_tracts = dc_final['census_tract_geoid'].notna().sum()
without_tracts = dc_final['census_tract_geoid'].isna().sum()

print(f"\nGeocoding Results:")
print(f"  Total facilities: {total}")
print(f"  ✓ With census tracts: {with_tracts} ({with_tracts/total*100:.1f}%)")
print(f"  ✗ Without census tracts: {without_tracts} ({without_tracts/total*100:.1f}%)")
print(f"  📍 Unique census tracts: {dc_final['census_tract_geoid'].nunique()}")

# Check for invalid GEOIDs (should be 11 digits)
if with_tracts > 0:
    dc_final['geoid_length'] = dc_final['census_tract_geoid'].astype(str).str.len()
    invalid_length = dc_final[dc_final['geoid_length'] != 11]['census_tract_geoid'].notna().sum()
    
    if invalid_length > 0:
        print(f"\n⚠️  WARNING: {invalid_length} records have invalid GEOID length")
        print(dc_final[dc_final['geoid_length'] != 11][['name', 'state_abb', 'census_tract_geoid']])
    else:
        print(f"\n✓ All GEOIDs are properly formatted (11 digits)")

# Facilities per tract distribution
tract_counts = dc_final['census_tract_geoid'].value_counts()

print(f"\nFacilities per Census Tract:")
print(f"  Tracts with 1 facility: {(tract_counts == 1).sum()}")
print(f"  Tracts with 2-5 facilities: {((tract_counts >= 2) & (tract_counts <= 5)).sum()}")
print(f"  Tracts with 6-10 facilities: {((tract_counts >= 6) & (tract_counts <= 10)).sum()}")
print(f"  Tracts with 10+ facilities: {(tract_counts > 10).sum()}")
print(f"  Maximum in one tract: {tract_counts.max()}")

# Show top concentrated tracts
print(f"\nTop 5 Most Concentrated Census Tracts:")
for i, (geoid, count) in enumerate(tract_counts.head(5).items(), 1):
    facilities = dc_final[dc_final['census_tract_geoid'] == geoid]
    state = facilities['state_abb'].iloc[0]
    county = facilities['county'].iloc[0] if 'county' in facilities.columns else 'Unknown'
    print(f"  {i}. {geoid} ({state}, {county}): {count} facilities")

CENSUS TRACT GEOCODING VALIDATION

Geocoding Results:
  Total facilities: 2536
  ✓ With census tracts: 2533 (99.9%)
  ✗ Without census tracts: 3 (0.1%)
  📍 Unique census tracts: 1235

⚠️  WARNING: 2533 records have invalid GEOID length
                                        name state_abb  census_tract_geoid
0                                    Verizon        NJ        3.402300e+10
1     Discover Financial Services New Albany        OH        3.904901e+10
2                         Google Data Center        NC        3.702703e+10
3                           Project Alluvion        IA        1.915301e+10
4                 Apple - Maiden Data Center        NC        3.703501e+10
...                                      ...       ...                 ...
2531                         Core Scientific        TX        4.838995e+10
2532                    Monclova Data Center        OH        3.909501e+10
2533                  Waterville Data Center       OH         3.909501e+10
2534          

In [8]:
# Cell: Fix GEOID Formatting and Rejoin

print("="*60)
print("FIXING GEOID FORMAT AND REJOINING")
print("="*60)

# Reload the datasets fresh
dc_final = pd.read_csv('/Users/zdiener/data-center-classification/data/processed/datacenters_with_census_tracts.csv')
communities = pd.read_csv('/Users/zdiener/data-center-classification/data/raw/2.0-communities.csv')

print(f"\nDatasets loaded:")
print(f"  Data centers: {len(dc_final)}")
print(f"  Communities: {len(communities)}")

# Fix community dataset GEOID
print("\n1. Formatting community census tract IDs...")
communities['census_tract_id'] = communities['Census tract 2010 ID'].astype(str).str.replace('.0', '').str.zfill(11)

print(f"   Sample formatted community IDs:")
print(f"   {communities['census_tract_id'].head(5).tolist()}")

# Fix data center GEOID - handle both string and float formats
print("\n2. Formatting data center census tract GEOIDs...")

def clean_geoid(geoid):
    """Clean and format GEOID to 11-digit string"""
    if pd.isna(geoid):
        return None
    
    # Convert to string and remove decimal point if present
    geoid_str = str(geoid).replace('.0', '')
    
    # Remove any other decimal points
    if '.' in geoid_str:
        geoid_str = geoid_str.split('.')[0]
    
    # Pad with zeros to 11 digits
    geoid_str = geoid_str.zfill(11)
    
    return geoid_str

dc_final['census_tract_id'] = dc_final['census_tract_geoid'].apply(clean_geoid)

print(f"   Sample formatted datacenter IDs:")
print(f"   {dc_final['census_tract_id'].dropna().head(5).tolist()}")

# Validate formatting
print("\n3. Validating GEOID format...")

# Check lengths
community_lengths = communities['census_tract_id'].str.len().value_counts()
dc_lengths = dc_final['census_tract_id'].dropna().str.len().value_counts()

print(f"\n   Community ID lengths:")
print(f"   {community_lengths}")

print(f"\n   Datacenter ID lengths:")
print(f"   {dc_lengths}")

# Check for non-numeric characters
community_numeric = communities['census_tract_id'].str.isdigit().all()
dc_numeric = dc_final['census_tract_id'].dropna().str.isdigit().all()

print(f"\n   All community IDs numeric? {community_numeric}")
print(f"   All datacenter IDs numeric? {dc_numeric}")

# Check overlap
community_tracts = set(communities['census_tract_id'])
datacenter_tracts = set(dc_final['census_tract_id'].dropna())
overlap = community_tracts & datacenter_tracts

print(f"\n4. Checking overlap...")
print(f"   Community dataset tracts: {len(community_tracts):,}")
print(f"   Data center tracts: {len(datacenter_tracts):,}")
print(f"   Overlapping tracts: {len(overlap):,}")
print(f"   Coverage rate: {len(overlap)/len(datacenter_tracts)*100:.1f}%")

if len(overlap) == 0:
    print("\n⚠️  Still no overlap! Let's investigate further...")
    
    # Show sample from each
    print(f"\n   Sample community IDs:")
    for id in list(community_tracts)[:5]:
        print(f"   '{id}' (length: {len(id)})")
    
    print(f"\n   Sample datacenter IDs:")
    for id in list(datacenter_tracts)[:5]:
        print(f"   '{id}' (length: {len(id)})")

FIXING GEOID FORMAT AND REJOINING

Datasets loaded:
  Data centers: 2536
  Communities: 74134

1. Formatting community census tract IDs...
   Sample formatted community IDs:
   ['01001020100', '01001020200', '01001020300', '01001020400', '01001020500']

2. Formatting data center census tract GEOIDs...
   Sample formatted datacenter IDs:
   ['34023000702', '39049007205', '37027030300', '19153011028', '37035011702']

3. Validating GEOID format...

   Community ID lengths:
   census_tract_id
11    74134
Name: count, dtype: int64

   Datacenter ID lengths:
   census_tract_id
11    2533
Name: count, dtype: int64

   All community IDs numeric? True
   All datacenter IDs numeric? True

4. Checking overlap...
   Community dataset tracts: 74,134
   Data center tracts: 1,235
   Overlapping tracts: 874
   Coverage rate: 70.8%


/var/folders/wy/tbdpl_451258rc4j_fzpf5x80000gn/T/ipykernel_76080/2054972900.py:9: DtypeWarning: Columns (0: Identified as disadvantaged due to tribal overlap, 1: Income data has been estimated based on geographic neighbor income, 2: Does the tract have at least 35 acres in it?, 3: Tract experienced historic underinvestment, 4: Is there at least one abandoned mine in this census tract?, 5: Names of Tribal areas within Census tract) have mixed types. Specify dtype option on import or set low_memory=False.
  communities = pd.read_csv('/Users/zdiener/data-center-classification/data/raw/2.0-communities.csv')
/var/folders/wy/tbdpl_451258rc4j_fzpf5x80000gn/T/ipykernel_76080/2054972900.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  communities['census_tract_i

In [10]:
# Cell: Load and Explore EJScreen 2024 Data

import pandas as pd
import zipfile

print("="*60)
print("LOADING EPA EJSCREEN 2024 DATA")
print("="*60)

# Path to your downloaded file
ejscreen_zip_path = '/Users/zdiener/Downloads/EJScreen 2024/EJScreen_2024_Tract_StatePct_with_AS_CNMI_GU_VI.csv.zip'

print("\nExtracting and loading EJScreen data...")

# Read directly from zip
with zipfile.ZipFile(ejscreen_zip_path, 'r') as zip_ref:
    # Get the CSV filename inside the zip
    csv_filename = zip_ref.namelist()[0]
    print(f"Found file: {csv_filename}")
    
    # Load the CSV
    with zip_ref.open(csv_filename) as file:
        ejscreen = pd.read_csv(file, low_memory=False)

print(f"✓ Loaded {len(ejscreen):,} census tracts")
print(f"✓ {len(ejscreen.columns)} columns")

# Display basic info
print(f"\n{'='*60}")
print("EJSCREEN DATASET OVERVIEW")
print(f"{'='*60}")

print(f"\nKey identifier columns:")
print(f"  ID: {ejscreen['ID'].dtype} (Census Tract GEOID)")
print(f"  Sample IDs: {ejscreen['ID'].head(3).tolist()}")

print(f"\nStates covered:")
print(ejscreen['ST_ABBREV'].value_counts().head(10))

print(f"\nColumn categories:")
# Group columns by prefix
demographics = [col for col in ejscreen.columns if col.startswith('ACSTOT') or col.startswith('MINORI') or col.startswith('LOW')]
environmental = [col for col in ejscreen.columns if col in ['PM25', 'OZONE', 'DSLPM', 'CANCER', 'RESP', 'PTRAF', 'PWDIS', 'PNPL', 'PRMP', 'PTSDF', 'UST', 'OVER64', 'PRE1960']]
percentiles = [col for col in ejscreen.columns if col.startswith('P_')]

print(f"  Demographics: {len(demographics)} columns")
print(f"  Environmental indicators: {len(environmental)} columns")
print(f"  Percentiles: {len(percentiles)} columns")

LOADING EPA EJSCREEN 2024 DATA

Extracting and loading EJScreen data...
Found file: EJScreen_2024_Tract_StatePct_with_AS_CNMI_GU_VI.csv
✓ Loaded 86,082 census tracts
✓ 230 columns

EJSCREEN DATASET OVERVIEW

Key identifier columns:
  ID: int64 (Census Tract GEOID)
  Sample IDs: [1001020100, 1001020200, 1001020300]

States covered:
ST_ABBREV
CA    9129
TX    6896
NY    5411
FL    5160
PA    3446
IL    3265
OH    3168
MI    3017
GA    2796
NC    2672
Name: count, dtype: int64

Column categories:
  Demographics: 5 columns
  Environmental indicators: 11 columns
  Percentiles: 50 columns


In [11]:
# Cell: Identify the 13 Environmental Burden Indicators

print("="*60)
print("13 ENVIRONMENTAL BURDEN INDICATORS")
print("="*60)

# The 13 EJScreen environmental indicators
environmental_indicators = {
    # Air Quality
    'DSLPM': 'Diesel Particulate Matter',
    'CANCER': 'Air Toxics Cancer Risk',
    'RESP': 'Air Toxics Respiratory HI',
    'PTRAF': 'Traffic Proximity',
    'PM25': 'Particulate Matter 2.5',
    'OZONE': 'Ozone',
    
    # Water & Waste
    'PWDIS': 'Wastewater Discharge',
    
    # Hazardous Materials
    'PNPL': 'Proximity to NPL (Superfund) Sites',
    'PRMP': 'Proximity to RMP Facilities',
    'PTSDF': 'Proximity to TSDF Facilities',
    
    # Built Environment
    'PRE1960': 'Lead Paint Indicator (% Pre-1960 Housing)',
    'PNPL': 'Underground Storage Tanks',
    
    # Additional Environmental
    'OVER64': 'Individuals over 64'
}

print("\nChecking which indicators are in the dataset:\n")

available_indicators = {}
for indicator, description in environmental_indicators.items():
    if indicator in ejscreen.columns:
        # Check for raw value and percentile
        raw_col = indicator
        pctile_col = f'P_{indicator}'
        
        print(f"✓ {indicator:10} | {description:35}")
        print(f"    Raw value:  {raw_col:15} {'[EXISTS]' if raw_col in ejscreen.columns else '[MISSING]'}")
        print(f"    Percentile: {pctile_col:15} {'[EXISTS]' if pctile_col in ejscreen.columns else '[MISSING]'}")
        
        available_indicators[indicator] = {
            'description': description,
            'raw_col': raw_col,
            'pctile_col': pctile_col
        }
        print()

print(f"\n✓ Found {len(available_indicators)} environmental indicators")

13 ENVIRONMENTAL BURDEN INDICATORS

Checking which indicators are in the dataset:

✓ DSLPM      | Diesel Particulate Matter          
    Raw value:  DSLPM           [EXISTS]
    Percentile: P_DSLPM         [EXISTS]

✓ PTRAF      | Traffic Proximity                  
    Raw value:  PTRAF           [EXISTS]
    Percentile: P_PTRAF         [EXISTS]

✓ PM25       | Particulate Matter 2.5             
    Raw value:  PM25            [EXISTS]
    Percentile: P_PM25          [EXISTS]

✓ OZONE      | Ozone                              
    Raw value:  OZONE           [EXISTS]
    Percentile: P_OZONE         [EXISTS]

✓ PWDIS      | Wastewater Discharge               
    Raw value:  PWDIS           [EXISTS]
    Percentile: P_PWDIS         [EXISTS]

✓ PNPL       | Underground Storage Tanks          
    Raw value:  PNPL            [EXISTS]
    Percentile: P_PNPL          [EXISTS]

✓ PRMP       | Proximity to RMP Facilities        
    Raw value:  PRMP            [EXISTS]
    Percentile: P_PRM

In [12]:
# Cell: Extract Relevant EJScreen Columns

print("="*60)
print("PREPARING EJSCREEN DATA FOR JOIN")
print("="*60)

# Select columns we need
columns_to_keep = [
    # Identifiers
    'ID',           # Census Tract GEOID
    'ST_ABBREV',    # State
    'CNTY_NAME',    # County
    
    # Demographics (for context)
    'ACSTOTPOP',    # Total Population
    'ACSTOTHH',     # Total Households
    'MINORPCT',     # Percent Minority
    'LOWINCPCT',    # Percent Low Income
    
    # 13 Environmental Indicators (Raw Values)
    'DSLPM',        # Diesel Particulate Matter
    'CANCER',       # Air Toxics Cancer Risk
    'RESP',         # Air Toxics Respiratory HI
    'PTRAF',        # Traffic Proximity
    'PM25',         # Particulate Matter 2.5
    'OZONE',        # Ozone
    'PWDIS',        # Wastewater Discharge
    'PNPL',         # Proximity to NPL Sites
    'PRMP',         # Proximity to RMP Facilities
    'PTSDF',        # Proximity to TSDF Facilities
    'PRE1960',      # Lead Paint Indicator
    'UST',          # Underground Storage Tanks
    'OVER64',       # Population Over 64
    
    # 13 Environmental Indicators (State Percentiles)
    'P_DSLPM',
    'P_CANCER',
    'P_RESP',
    'P_PTRAF',
    'P_PM25',
    'P_OZONE',
    'P_PWDIS',
    'P_PNPL',
    'P_PRMP',
    'P_PTSDF',
    'P_PRE1960',
    'P_UST',
    'P_OVER64',
    
    # EJ Index columns (combine env + demographics)
    'P_LDPNT',      # EJ Index for Traffic
    'P_DSLPM_D2',   # EJ Index for Diesel PM
    'P_PM25_D2',    # EJ Index for PM2.5
]

# Filter to only columns that exist
columns_to_keep = [col for col in columns_to_keep if col in ejscreen.columns]

print(f"\nSelecting {len(columns_to_keep)} columns from EJScreen")

ejscreen_subset = ejscreen[columns_to_keep].copy()

# Rename ID to census_tract_id for clarity
ejscreen_subset = ejscreen_subset.rename(columns={'ID': 'census_tract_id'})

# Ensure census_tract_id is string and properly formatted
ejscreen_subset['census_tract_id'] = ejscreen_subset['census_tract_id'].astype(str).str.zfill(11)

print(f"\n✓ EJScreen subset prepared")
print(f"  Records: {len(ejscreen_subset):,}")
print(f"  Columns: {len(ejscreen_subset.columns)}")

# Show sample
print(f"\nSample data:")
print(ejscreen_subset.head(3))

PREPARING EJSCREEN DATA FOR JOIN

Selecting 27 columns from EJScreen

✓ EJScreen subset prepared
  Records: 86,082
  Columns: 27

Sample data:
  census_tract_id ST_ABBREV       CNTY_NAME  ACSTOTPOP  ACSTOTHH  LOWINCPCT  \
0     01001020100        AL  Autauga County     1865.0     700.0   0.256836   
1     01001020200        AL  Autauga County     1861.0     544.0   0.262433   
2     01001020300        AL  Autauga County     3492.0    1305.0   0.279403   

      DSLPM          PTRAF      PM25     OZONE       PWDIS  PNPL  PRMP  PTSDF  \
0  0.087407  244690.919118  8.969069  50.63864  205.824356   0.0   0.0    0.0   
1  0.093301  365028.762166  8.998258  50.65087  245.727346   0.0   0.0    0.0   
2  0.093484  406914.547835  9.035765  50.76981  264.202236   0.0   0.0    0.0   

   PRE1960       UST  OVER64  P_DSLPM  P_PTRAF  P_PM25  P_OZONE  P_PWDIS  \
0    189.0  0.330720   363.0     45.0     48.0    63.0      5.0     61.0   
1    174.0  2.208384   293.0     49.0     55.0    65.0      5.0

In [13]:
# Cell: Join Data Centers with EJScreen

print("="*60)
print("JOINING DATA CENTERS WITH EJSCREEN")
print("="*60)

# Load your geocoded data centers
dc_final = pd.read_csv('/Users/zdiener/data-center-classification/data/processed/datacenters_with_census_tracts.csv')

print(f"\nDatasets:")
print(f"  Data centers: {len(dc_final):,}")
print(f"  EJScreen tracts: {len(ejscreen_subset):,}")

# Format census tract IDs (remove .0 if present from float)
dc_final['census_tract_id'] = dc_final['census_tract_geoid'].astype(str).str.replace('.0', '').str.replace('nan', '').str.zfill(11)

# Remove invalid IDs
dc_final.loc[dc_final['census_tract_id'] == '00000000000', 'census_tract_id'] = None

# Check overlap
ejscreen_tracts = set(ejscreen_subset['census_tract_id'])
datacenter_tracts = set(dc_final['census_tract_id'].dropna())
overlap = ejscreen_tracts & datacenter_tracts

print(f"\nTract Coverage:")
print(f"  EJScreen tracts: {len(ejscreen_tracts):,}")
print(f"  Data center tracts: {len(datacenter_tracts):,}")
print(f"  Overlapping tracts: {len(overlap):,}")
print(f"  Coverage rate: {len(overlap)/len(datacenter_tracts)*100:.1f}%")

# Show sample IDs for debugging
if len(overlap) == 0:
    print(f"\n⚠️  No overlap found!")
    print(f"\nSample EJScreen IDs: {list(ejscreen_tracts)[:5]}")
    print(f"Sample Datacenter IDs: {list(datacenter_tracts)[:5]}")
else:
    print(f"\n✓ Good overlap - proceeding with join")

# Perform the join
print(f"\nPerforming join...")
dc_with_ejscreen = dc_final.merge(
    ejscreen_subset,
    on='census_tract_id',
    how='left',
    indicator=True,
    suffixes=('', '_ejscreen')
)

# Summary
matched = (dc_with_ejscreen['_merge'] == 'both').sum()
unmatched = (dc_with_ejscreen['_merge'] == 'left_only').sum()

print(f"\nJoin Results:")
print(f"  Total facilities: {len(dc_final):,}")
print(f"  ✓ Successfully joined: {matched:,} ({matched/len(dc_final)*100:.1f}%)")
print(f"  ✗ No EJScreen data: {unmatched:,} ({unmatched/len(dc_final)*100:.1f}%)")

if unmatched > 0:
    print(f"\n  Unmatched facilities by state:")
    unmatched_df = dc_with_ejscreen[dc_with_ejscreen['_merge'] == 'left_only']
    print(unmatched_df['state_abb'].value_counts().head(10))

# Save final dataset
output_path = '/Users/zdiener/data-center-classification/data/processed/datacenters_with_ejscreen_final.csv'
dc_with_ejscreen.to_csv(output_path, index=False)
print(f"\n✓ Saved final dataset to: {output_path}")

print(f"\n{'='*60}")
print("DATASET READY FOR CLASSIFICATION!")
print(f"{'='*60}")
print(f"\nFinal dataset includes:")
print(f"  ✓ {len(dc_with_ejscreen):,} data center facilities")
print(f"  ✓ 13 environmental burden indicators")
print(f"  ✓ State percentiles for comparison")
print(f"  ✓ Demographics (population, minority %, low income %)")
print(f"  ✓ Geographic identifiers (state, county, census tract)")

JOINING DATA CENTERS WITH EJSCREEN

Datasets:
  Data centers: 2,536
  EJScreen tracts: 86,082

Tract Coverage:
  EJScreen tracts: 86,082
  Data center tracts: 1,235
  Overlapping tracts: 1,227
  Coverage rate: 99.4%

✓ Good overlap - proceeding with join

Performing join...

Join Results:
  Total facilities: 2,536
  ✓ Successfully joined: 2,525 (99.6%)
  ✗ No EJScreen data: 11 (0.4%)

  Unmatched facilities by state:
state_abb
CT    8
ME    1
AK    1
AL    1
Name: count, dtype: int64

✓ Saved final dataset to: /Users/zdiener/data-center-classification/data/processed/datacenters_with_ejscreen_final.csv

DATASET READY FOR CLASSIFICATION!

Final dataset includes:
  ✓ 2,536 data center facilities
  ✓ 13 environmental burden indicators
  ✓ State percentiles for comparison
  ✓ Demographics (population, minority %, low income %)
  ✓ Geographic identifiers (state, county, census tract)
